# Diversity-led selection on Kaggle (both T4 GPUs)

Trains the new label-free selection methods on NYUv2 (RGB-D, 4000 iters, seeds 0-2, same settings as every earlier run).
Two runs train at the same time, one per GPU, so a part takes about 4 to 5 hours.

**Before the first run**
1. **Add Input** → your `rgbx-data` dataset (the one with `nyuv2_prepared.zip`). No weights are needed.
2. **Settings** → Accelerator **GPU T4 x2**, Internet **on**.

**Each part**: set `PART` in cell 4, then **Save Version → Save & Run All (Commit)**. It runs on Kaggle's servers, so the laptop can be off.
Do PART 1 first (the main comparison), then PART 2 (baselines and ablations).

**What to paste back**: the block at the end of the last cell's output, from `===== PASTE FROM HERE =====` to `===== PASTE TO HERE =====`.
Also download `runs/results.csv` from the version's **Output** tab, just in case.

In [ ]:
# Cell 1: check both GPUs, get the code
import os, glob, subprocess, time, csv, torch
def sh(cmd):
    print('$', cmd, flush=True); return subprocess.run(cmd, shell=True).returncode

N_GPU = torch.cuda.device_count()
print(torch.__version__, N_GPU, 'GPU(s):', [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert N_GPU >= 1, 'Turn on the GPU: Settings > Accelerator > GPU T4 x2'
torch.ones(1).cuda().sum().item()   # an error here means this GPU is not supported: choose GPU T4 x2
if N_GPU < 2:
    print('WARNING: only one GPU, so runs go one at a time and a part takes about twice as long')

CODE, BRANCH = '/tmp/code', 'claude/project-thread-euud8r'
sh(f'rm -rf {CODE} && git clone -q -b {BRANCH} https://github.com/talha-ansarii/btech-project.git {CODE}')
sh(f'cd {CODE} && git log --oneline -1')
sh(f'pip -q install -r {CODE}/requirements.txt')

In [ ]:
# Cell 2: find the data in your Kaggle dataset
INPUT = '/kaggle/input'
def find(name):
    hits = glob.glob(f'{INPUT}/**/{name}', recursive=True) + glob.glob(f'/tmp/unz/**/{name}', recursive=True)
    if not hits:   # Kaggle sometimes keeps uploaded zips as zips: unpack them once and look again
        for z in glob.glob(f'{INPUT}/**/*.zip', recursive=True):
            sh(f'mkdir -p /tmp/unz && unzip -q -o "{z}" -d /tmp/unz')
        hits = glob.glob(f'/tmp/unz/**/{name}', recursive=True)
    assert hits, f'{name} not found: add your rgbx-data dataset with "Add Input"'
    return os.path.dirname(hits[0])

DATA = find('meta.json')
WORK = '/kaggle/working'
CACHE, SPLITS, RUNS = f'{WORK}/cache', f'{WORK}/splits_div', f'{WORK}/runs'
print('DATA', DATA)
print(len(os.listdir(f'{DATA}/train/rgb')), len(os.listdir(f'{DATA}/test/rgb')), '(expect 795 654)')

In [ ]:
# Cell 3: label-free scores and features (~5 min), then all selected subsets and what they contain
sh(f'python {CODE}/complementarity.py --data {DATA} --split train --out {CACHE}/labelfree_train.npz')
sh(f'python {CODE}/features_rgbd.py --data {DATA} --out {CACHE}/feats_rgbd.npz')
METHODS = 'kmeans typiclust kmeans-rgbd cover-rgbd cover-rgb kmeans-ctb'
for frac in ('0.05', '0.10'):
    sh(f'python {CODE}/select_subsets.py --cache {CACHE} --rgbd-feats {CACHE}/feats_rgbd.npz '
       f'--out-dir {SPLITS} --frac {frac} --methods {METHODS}')
print(len(os.listdir(SPLITS)), 'split files (expect 36)')
sh(f'python {CODE}/analyze_splits.py --data {DATA} --splits {CODE}/splits {SPLITS} --md {WORK}/split_stats.md')

In [ ]:
# Cell 4: train. Set PART, then Save Version > Save & Run All (Commit)
PART = 1   # 1 the first time, then change to 2 and save a new version
JOBS = {
    # main comparison: the two RGB-D diversity methods at both budgets, and plain kmeans at 10% (15 runs)
    1: [('cover-rgbd', '0.05'), ('kmeans-rgbd', '0.05'), ('cover-rgbd', '0.10'), ('kmeans-rgbd', '0.10'),
        ('kmeans', '0.10')],
    # published baseline and ablations (12 runs)
    2: [('typiclust', '0.05'), ('typiclust', '0.10'), ('cover-rgb', '0.05'), ('kmeans-ctb', '0.05')],
}
queue = [(m, f, s) for m, f in JOBS[PART] for s in (0, 1, 2)]
n_q = max(1, min(2, N_GPU))
procs = []
for g in range(n_q):
    mine = queue[g::n_q]
    # one shell line per GPU: its runs one after another, logged to a file
    cmds = ' ; '.join(f'python {CODE}/run_selection.py --data {DATA} --out {RUNS}/gpu{g} --splits-dir {SPLITS} '
                      f'--jobs {m}:{f} --seeds {s}' for m, f, s in mine)
    log = open(f'{WORK}/train_gpu{g}.log', 'w')
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    procs.append(subprocess.Popen(cmds, shell=True, stdout=log, stderr=subprocess.STDOUT, env=env))
    print(f'GPU {g}: {len(mine)} runs:', ', '.join(f'{m}{f} s{s}' for m, f, s in mine))

t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(300)
    done = []
    for g in range(n_q):
        done += [l.strip() for l in open(f'{WORK}/train_gpu{g}.log') if l.startswith('FINAL')]
    print(f'{(time.time() - t0) / 60:.0f} min: {len(done)}/{len(queue)} runs finished', flush=True)
for g in range(n_q):
    print(f'--- last lines of GPU {g} log ---'); sh(f'tail -n 3 {WORK}/train_gpu{g}.log')

In [ ]:
# Cell 5: merge both GPUs' results, summary table, clean up big files
rows, header = [], None
for f in sorted(glob.glob(f'{RUNS}/gpu*/results.csv')):
    with open(f) as fh:
        r = list(csv.reader(fh)); header = r[0]; rows += r[1:]
os.makedirs(RUNS, exist_ok=True)
with open(f'{RUNS}/results.csv', 'w', newline='') as fh:
    w = csv.writer(fh); w.writerow(header or ['run']); w.writerows(rows)
sh(f'python {CODE}/summarize.py --csv {RUNS}/results.csv --md {RUNS}/summary.md > /dev/null')
sh(f'find {RUNS} -name best.pt -delete')   # keep final.pt only, so the output stays small enough to download

print('===== PASTE FROM HERE =====')
print(f'PART {PART}, {len(rows)} of {len(queue)} runs finished, commit {subprocess.getoutput(f"cd {CODE} && git log --oneline -1")}')
print(open(f'{RUNS}/results.csv').read())
print(open(f'{WORK}/split_stats.md').read())
print('===== PASTE TO HERE =====')